# 25. XGBoost seed sweep

**One variable against ledger row 38** (`xgb_te`, CV 0.967099): the model's random
seed. Everything else is held, including the encoder's inner split, so only the
learner's own stochasticity varies.

## Why this exists

Row 38 measured XGBoost at **+0.000316 over row 17, paired sd 0.000134, 5/5 folds**,
which makes it the best single model in this repo. It was one seed, and this repo has
a binding precedent for what that is worth on its own.

Row 26 measured CatBoost at +0.000132, 5/5 folds, t(4)=3.72, and **logged it as parity
rather than as an improvement**, because this repo requires a gain to hold across at
least two seeds and only one had been run. Rows 28 to 31 then ran four more and
resolved it upward: the five CatBoost seeds spanned 1.32e-05 against the five LightGBM
target-encoded seeds' 6.00e-05, and the family means differed by +0.000157 at 13.4
standard errors, which one seed could not have produced by chance.

This is that same step for XGBoost, and it is run for the same reason.

## What it can and cannot settle

It sizes XGBoost's seed spread and puts a standard error on the family mean, so it can
say whether +0.000316 is a property of the learner or of seed 42. It cannot say
anything about the stack, which is a separate notebook and a separate ledger row.

**The seed-42 arm is row 38's configuration exactly**, so it doubles as this run's
reproduction check. If it does not reproduce row 38, the harness is not row 38's
harness and no arm in the sweep is comparable to it.

## Cost, and why five

Five seeds matches the CatBoost sweep, which is the only thing XGBoost's spread has to
be compared against. Fewer would answer the "is it real" question but would not make
the two spreads comparable, and the encoder is built once per fold and shared by every
arm, so the marginal cost of an arm is training alone.

In [ ]:
# One flag. The sweep always runs top to bottom on Kaggle.
SMOKE = True

SEED = 42
N_INNER = 5
SMOOTH = 10.0

# The knob under test. 42 is row 38's seed, so that arm is the reproduction check.
SEEDS = [42, 2024, 7, 2025, 13]
BASE_SEED = 42

# Row 38's configuration, held.
LR = 0.05
N_EST = 2000
BENCH_EST = 200
PROBE_FOLD = 0
MAX_DEPTH = 6
N_JOBS = -1

# Ledger row 38, this configuration at seed 42.
BASELINE_NAME = "xgb_te"
BASELINE_CV = 0.967099
EXPECTED_FOLD_SHA = "ec282b0968059676"

# For the family comparison. Rows 17 and 26, the other two learners on this encoder,
# and the two seed spreads already measured in this repo.
LGBM_CV = 0.966782
CATBOOST_CV = 0.966915
CATBOOST_SPREAD = 1.32e-05
LGBM_TE_SPREAD = 6.00e-05

EXPECTED_LEAK2 = 8.1e-05
EXPECTED_PRIOR_SHIFT = 1.3e-04

print(f"SMOKE = {SMOKE}   seeds {SEEDS}   base {BASE_SEED}")

## Stage 1. Data, folds, leak checklist

The fold checksum is the only thing standing between an out-of-fold vector that
blends and one that is silently misaligned, so it is checked before anything trains
rather than after.

In [ ]:
import ast
import gc
import hashlib
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold

# Runs here or on Kaggle. Both are found by name rather than by assuming a shape.
KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
SUB = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "submissions"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
TARGET = "addicted_label"
CAT = ["gender", "stress_level", "academic_work_impact"]
COLS = [c for c in train_full.columns if c not in ("id", TARGET)]

# Leak checklist, re-run rather than ticked by inspection. `id` is a contiguous row
# index that separates train from test perfectly, so it is a guaranteed leak if it
# ever reaches the model.
checks = {
    "id is not a feature": "id" not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full["id"]) & set(test["id"])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != "id"],
}
for name, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")
LEAK_OK = all(checks.values())

# ROW_IDX maps this run's rows back into the saved member vectors.
if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
    N_EST, BENCH_EST = 200, 50
    # Measured 2026-08-19 and written up: at 16,000 rows this machine
    # runs 101x slower at n_jobs=-1 than at n_jobs=1, monotone in the thread count.
    # N_JOBS above is chosen to match row 17 on Kaggle at 691,369 rows, where it is
    # right. A smoke run produces no ledger number, so overriding it here costs
    # nothing and is the difference between two minutes and giving up on the check.
    N_JOBS = 1
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy()
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

sha = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
ALIGNED = sha == EXPECTED_FOLD_SHA
print()
print(f"rows {len(train):,}   target rate {y.mean():.6f}")
print(f"fold sizes {np.bincount(folds).tolist()}")
print(f"fold sha {sha}  expected {EXPECTED_FOLD_SHA}")
if SMOKE:
    print("SMOKE: subsampled, so the sha is EXPECTED to differ. Not a check.")
else:
    print("fold alignment: VERIFIED" if ALIGNED else
          "fold alignment: MISMATCH - the OOF from this run is not blendable")

X = train[COLS].copy()
X_test = test[COLS].copy()
for c in CAT:
    X[c] = X[c].astype("category")
    X_test[c] = X_test[c].astype("category")

## Stage 2. The encoder

Copied from `13_target_encoding.ipynb` so the feature set is row 17's feature set. A
copy is a provenance risk under the notebook layout, so it is checked rather than
asserted: the cell below parses the encoder out of `13`, normalises both versions
through `ast.unparse`, and compares checksums. Expected fingerprint
`0642e41750ef8bab`, the same value rows 26 and 33 recorded.

In [ ]:
def _stats(levels, yy, prior):
    """Smoothed target mean and level frequency, fit only on the rows given."""
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    return mean, g["count"] / len(df)


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    # A level unseen while fitting falls back to the prior and zero mass.
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt=None, seed=SEED):
    """Encodings for ONE outer fold: (train, valid, test)."""
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))

    for c in COLS:
        lv = Xf[c].to_numpy()
        # Fit once on the whole training portion, for validation and test rows.
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        if Xt is not None:
            e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean,
                                                      freq, prior)
        # Training rows get inner out-of-fold values.
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if Xt is not None else None)


def build(Xf, yy, tr, va, Xt=None, seed=SEED):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt, seed)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = None if Xt is None else pd.concat(
        [Xt.reset_index(drop=True), d_te], axis=1)
    return Xtr, Xva, Xte


ENCODER_FNS = ("_stats", "_apply", "encode_fold", "build")


def fingerprint(src):
    """Semantic checksum of the encoder functions inside a block of source."""
    body = ast.parse(src).body
    parts = [ast.unparse(n) for n in body
             if isinstance(n, ast.FunctionDef) and n.name in ENCODER_FNS]
    if len(parts) != len(ENCODER_FNS):
        return None
    return hashlib.sha256("\n".join(parts).encode()).hexdigest()[:16]


import inspect

mine = fingerprint("\n".join(inspect.getsource(f)
                             for f in (_stats, _apply, encode_fold, build)))

theirs = None
try:
    src13 = locate("13_target_encoding.ipynb")
except FileNotFoundError:
    src13 = None
if src13 is not None:
    import json as _json
    for c in _json.loads(src13.read_text(encoding="utf-8"))["cells"]:
        if c["cell_type"] == "code" and "def encode_fold" in "".join(c["source"]):
            theirs = fingerprint("".join(c["source"]))
            break

ENCODER_MATCH = mine is not None and mine == theirs
print(f"encoder fingerprint here        : {mine}")
print(f"encoder fingerprint in 13       : {theirs}")
print(f"rows 26 and 33 recorded         : 0642e41750ef8bab")
print("encoder: IDENTICAL to row 17's" if ENCODER_MATCH else
      "encoder: DIFFERS from 13 (or 13 not found) - this is NOT one variable")
print()
print(f"{len(COLS)} raw columns -> {len(COLS) * 3} features after encoding")

### The leak checks, by execution

The same three checks `13` ran, on the same encoder, so their numbers are directly
comparable to the ones recorded earlier. Read all three together: the first two must be
about zero, the third must be large. Without the third, an encoder that ignored the
target entirely would pass the first two and look clean.

In [ ]:
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
d_tr0, d_va0, _ = encode_fold(X, y, _tr, _va)

# 1. A validation row's own target must never reach its own encoding.
y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_fold(X, y1, _tr, _va)
leak1 = max(np.abs(d_va0[f"te_{c}"] - d_va1[f"te_{c}"]).max() for c in COLS)

# 2. A training row's own target must never reach its own inner encoding. A small
# residual is expected and is not a leak: `prior` is the training-portion mean.
_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_fold(X, y2, _tr, _va)
leak2 = max(np.abs(d_tr0[f"te_{c}"].to_numpy()[pick]
                   - d_tr2[f"te_{c}"].to_numpy()[pick]).max() for c in COLS)
prior_shift = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

# 3. The encoding MUST move when targets it is allowed to see change.
y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_fold(X, y3, _tr, _va)
live = max(np.abs(d_va0[f"te_{c}"] - d_va3[f"te_{c}"]).max() for c in COLS)

print(f"1. flip all validation targets -> change in their encoding: {leak1:.3e}")
print(f"2. flip 200 training rows -> change in their own encoding:  {leak2:.3e}")
print(f"   prior moved {prior_shift:.3e}, and these should track each other")
print(f"3. flip all training targets -> change in val encoding:     {live:.3e}")

CLEAN = leak1 == 0 and leak2 < 10 * max(prior_shift, 1e-9) and live > 0.1
print()
print("LEAK CHECKS: PASS" if CLEAN else "LEAK CHECKS: FAILED - do not log this run")
if not SMOKE:
    print(f"13 recorded leak2 {EXPECTED_LEAK2:.1e} against a prior shift of "
          f"{EXPECTED_PRIOR_SHIFT:.1e}; this run gives {leak2:.1e} and "
          f"{prior_shift:.1e}")

del d_tr0, d_va0, d_va1, d_tr2, d_va3, y1, y2, y3
gc.collect()

## Stage 3. Bench and determinism

Identical to row 38's configuration in every respect except that the seed is now a
parameter. The determinism check trains the same seed twice inside this one kernel and
requires bit-identical predictions, which is what separates a seed effect from
run-to-run noise.

In [ ]:
import xgboost as xgb


def make(n_est, seed):
    return xgb.XGBClassifier(
        objective="binary:logistic", eval_metric="auc",
        tree_method="hist", enable_categorical=True,
        learning_rate=LR, n_estimators=n_est, max_depth=MAX_DEPTH,
        subsample=0.8, colsample_bytree=0.8,
        random_state=seed, n_jobs=N_JOBS, verbosity=0,
    )


def fit_arm(Xtr, ytr, Xva, seed, n_est, Xte=None):
    m = make(n_est, seed)
    t0 = time.time()
    m.fit(Xtr, ytr)
    secs = time.time() - t0
    p = m.predict_proba(Xva)[:, 1]
    p_te = m.predict_proba(Xte)[:, 1] if Xte is not None else None
    return p, p_te, secs


def hhmm(s):
    return f"{int(s // 60)}m {int(s % 60):02d}s"


LOG = (Path("/kaggle/working") if ON_KAGGLE
       else LOCAL / "artifacts" / "logs") / "25_xgb_seeds.log"
LOG.parent.mkdir(parents=True, exist_ok=True)


def note(msg):
    print(msg)
    with LOG.open("a", encoding="utf-8") as fh:
        print(f"{time.strftime('%H:%M:%S')}  {msg}", file=fh, flush=True)


note(f"=== run start, SMOKE={SMOKE}, seeds={SEEDS}, n_jobs={N_JOBS} ===")

_t0 = time.time()
tr0 = np.where(folds != PROBE_FOLD)[0]
va0 = np.where(folds == PROBE_FOLD)[0]
Xtr0, Xva0, _ = build(X, y, tr0, va0)
ENC_SECS = time.time() - _t0
print(f"encoder, one fold: {hhmm(ENC_SECS)}   {Xtr0.shape[1]} features")

pa, _, sa = fit_arm(Xtr0, y[tr0], Xva0, BASE_SEED, BENCH_EST)
pb, _, _ = fit_arm(Xtr0, y[tr0], Xva0, BASE_SEED, BENCH_EST)
delta = float(np.abs(pa - pb).max())
DETERMINISTIC = delta == 0.0

# A different seed MUST move the predictions. Without this, a silently ignored seed
# would produce five identical arms and a spread of exactly zero, which would read as
# a spectacular stability result rather than as a bug.
pc, _, _ = fit_arm(Xtr0, y[tr0], Xva0, SEEDS[1], BENCH_EST)
SEED_LIVE = float(np.abs(pa - pc).max()) > 0

print(f"xgboost {xgb.__version__}, hist, n_jobs={N_JOBS}")
print(f"{BENCH_EST} trees at seed {BASE_SEED}: {hhmm(sa)}, "
      f"AUC {roc_auc_score(y[va0], pa):.6f}")
print(f"determinism, same seed twice, max |diff|: {delta:.3e}  "
      f"{'OK' if DETERMINISTIC else 'NOT REPRODUCIBLE'}")
print(f"seed {SEEDS[1]} moves the predictions: {'yes' if SEED_LIVE else 'NO - THE '
      'SEED IS BEING IGNORED, the sweep would be five copies of one model'}")

per_tree = sa / BENCH_EST
total = 5 * (ENC_SECS + per_tree * N_EST * len(SEEDS))
print()
print(f"projection, {N_EST} trees, five folds, {len(SEEDS)} seeds: {hhmm(total)}")
note(f"stage 3 done, determinism {'OK' if DETERMINISTIC else 'FAILED'}, "
     f"seed live {SEED_LIVE}, projected {hhmm(total)}")

del Xtr0, Xva0, pa, pb, pc
gc.collect()

## Stage 4. The sweep

The encoder is built once per fold and every seed trains on the identical matrices, so
the arms cannot differ through the encoder's inner split. Only the learner's own
stochasticity varies, which is the definition this repo used for the CatBoost and
LightGBM seed sweeps.

In [ ]:
oof = {s: np.zeros(len(train)) for s in SEEDS}
test_pred = {s: np.zeros(len(test)) for s in SEEDS}
per_fold = {s: [] for s in SEEDS}

t0 = time.time()
for f in range(5):
    tr = np.where(folds != f)[0]
    va = np.where(folds == f)[0]
    te0 = time.time()
    Xtr, Xva, Xte = build(X, y, tr, va, X_test)
    note(f"fold {f}: encoded in {hhmm(time.time() - te0)}")

    for s in SEEDS:
        p, p_te, secs = fit_arm(Xtr, y[tr], Xva, s, N_EST, Xte)
        oof[s][va] = p
        test_pred[s] += p_te / 5
        per_fold[s].append(float(roc_auc_score(y[va], p)))
        note(f"  fold {f} seed {s:>5}: {per_fold[s][-1]:.6f}  ({hhmm(secs)})")

    del Xtr, Xva, Xte
    gc.collect()
    done = time.time() - t0
    note(f"fold {f} done, elapsed {hhmm(done)}, "
         f"about {hhmm(done / (f + 1) * (4 - f))} left")

cv = {s: float(np.mean(per_fold[s])) for s in SEEDS}
sd = {s: float(np.std(per_fold[s])) for s in SEEDS}

print()
print(f"{'seed':>7} {'CV':>10} {'fold sd':>10}")
for s in SEEDS:
    star = "  <- row 38's seed" if s == BASE_SEED else ""
    print(f"{s:>7} {cv[s]:>10.6f} {sd[s]:>10.6f}{star}")
note(f"sweep done in {hhmm(time.time() - t0)}, "
     + ", ".join(f"{s}:{cv[s]:.6f}" for s in SEEDS))

### Does this kernel reproduce row 38?

The seed-42 arm is row 38's configuration exactly, so it has a number it is required to
hit. If it misses, the harness is not row 38's harness and nothing in the sweep is
comparable to it.

In [ ]:
repro = cv[BASE_SEED] - BASELINE_CV
REPRODUCED = abs(repro) < 1e-4

print(f"arm seed={BASE_SEED}: {cv[BASE_SEED]:.6f}")
print(f"ledger row 38      : {BASELINE_CV:.6f}")
print(f"difference         : {repro:+.2e}   "
      f"{'inside' if REPRODUCED else 'OUTSIDE'} the 1e-04 tolerance")
if SMOKE:
    print()
    print("SMOKE: a subsampled run against a full-data ledger number, so this is")
    print("EXPECTED to be far outside tolerance and is NOT a check here.")

if not SMOKE:
    # Row 38's vector is a kernel output rather than a dataset file, so it is usually
    # not mounted here. When it is, this is the strictest form of the check above.
    try:
        v = np.load(locate("xgb_te_oof.npy"))
        print(f"saved row 38 vector, max |diff| against this arm: "
              f"{np.abs(v - oof[BASE_SEED]).max():.3e}")
    except FileNotFoundError:
        print("(row 38's vector is not mounted here, so the CV comparison above is")
        print(" the check; it is compared per prediction when this runs locally)")

### The family, and what the spread says

The question row 38 could not answer: is +0.000316 a property of XGBoost on this
representation, or of seed 42. The family mean and its standard error answer it, and
the spread is directly comparable to the two seed sweeps already in this repo.

In [ ]:
vals = np.array([cv[s] for s in SEEDS])
spread = float(vals.max() - vals.min())
fam_mean = float(vals.mean())
fam_se = float(vals.std(ddof=1) / np.sqrt(len(vals)))

print(f"{'':34} {'mean':>10} {'spread':>10}")
print(f"{'XGBoost, this sweep':34} {fam_mean:>10.6f} {spread:>10.2e}")
print(f"{'CatBoost, rows 26 and 28 to 31':34} {CATBOOST_CV:>10.6f} "
      f"{CATBOOST_SPREAD:>10.2e}")
print(f"{'LightGBM TE, rows 17 and 20 to 23':34} {LGBM_CV:>10.6f} "
      f"{LGBM_TE_SPREAD:>10.2e}")
print()
print(f"XGBoost family mean {fam_mean:.6f} +/- {fam_se:.6f} (se over {len(SEEDS)} seeds)")
print(f"  vs LightGBM row 17 : {fam_mean - LGBM_CV:+.6f}"
      f"   ({(fam_mean - LGBM_CV) / fam_se:.1f} se)" if fam_se else "")
print(f"  vs CatBoost row 26 : {fam_mean - CATBOOST_CV:+.6f}"
      f"   ({(fam_mean - CATBOOST_CV) / fam_se:.1f} se)" if fam_se else "")
print()
print("Rows 28 to 31 put the CatBoost family +0.000157 above LightGBM at 13.4 se, and")
print("called that a resolution upward of row 26's parity claim. The same arithmetic")
print("applies here and is printed above rather than asserted.")

blocked = None
if not (LEAK_OK and CLEAN):
    blocked = "a leak check failed"
elif not ENCODER_MATCH:
    blocked = "the encoder does not match 13"
elif not DETERMINISTIC:
    blocked = "the configuration is not reproducible"
elif not SEED_LIVE:
    blocked = "the seed is being ignored, so these are not five models"
elif not SMOKE and not ALIGNED:
    blocked = "fold alignment failed, so these vectors are not blendable"
elif not SMOKE and not REPRODUCED:
    blocked = f"the seed {BASE_SEED} arm missed row 38 by {repro:+.2e}"

print()
if blocked:
    print(f"VERDICT: blocked, {blocked}")
elif SMOKE:
    print("SMOKE: no verdict. Subsampled rows cannot resolve a spread of this size.")
else:
    print(f"XGBoost is the strongest single-model family on this representation"
          if fam_mean > CATBOOST_CV else
          f"XGBoost family mean sits below CatBoost's single measured seed")
    print("The stack decision is a separate notebook and a separate ledger row.")

In [ ]:
pre = "SMOKE_" if SMOKE else ""
for s in SEEDS:
    np.save(OUT / f"{pre}xgb_te_seed{s}_oof.npy", oof[s])
    np.save(OUT / f"{pre}xgb_te_seed{s}_test.npy", test_pred[s])
print(f"wrote {pre}xgb_te_seed<n>_oof.npy and _test.npy for {SEEDS}")

print()
print("ledger lines, one per seed:")
for s in SEEDS:
    print(f"  xgb_te_seed{s:<5}  cv_mean {cv[s]:.6f}  cv_std {sd[s]:.6f}")
print()
print(f"  leak checks {'PASS' if CLEAN else 'FAILED'}, "
      f"fold alignment {'verified' if ALIGNED else 'NOT verified'}, "
      f"encoder {'matches 13' if ENCODER_MATCH else 'DIFFERS'}, "
      f"determinism {'OK' if DETERMINISTIC else 'FAILED'}, "
      f"seed live {'yes' if SEED_LIVE else 'NO'}")